# EDA on DPIRD NetCDF

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import numpy as np
import pandas as pd
import xarray as xr


DPIRD file finder

In [12]:
"""
Traverses up the directory tree to find the project root 
by looking for a specific marker file or folder.
"""
def get_project_root(current_dir=Path.cwd(), marker="dataset_DPIRD_utc0_clean"):
    for parent in [current_dir] + list(current_dir.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find project root with marker: {marker}")

project_root = get_project_root()
DPIRD_filepath = project_root / "dataset_DPIRD_utc0_clean" / "DPIRD_final_stations_utc0.nc"


In [14]:
ds= xr.load_dataset(DPIRD_filepath,engine='h5netcdf')
display(ds)

<xarray.Dataset> Size: 3GB
Dimensions:                       (station: 192, time: 105248)
Coordinates:
  * station                       (station) <U22 17kB 'Floreat Park' ... 'Gut...
  * time                          (time) datetime64[ns] 842kB 2022-01-01 ... ...
    lat                           (station) float64 2kB -31.95 -33.77 ... -29.06
    lon                           (station) float64 2kB 115.8 120.5 ... 115.8
    code                          (station) <U5 4kB 'FL' 'JP001' ... 'GU001'
Data variables: (12/18)
    airTemperature                (station, time) float64 162MB 20.4 ... 39.4
    apparentAirTemperature        (station, time) float64 162MB 18.3 ... 38.9
    relativeHumidity              (station, time) float64 162MB 48.9 ... 18.9
    dewPoint                      (station, time) float64 162MB 9.3 9.3 ... 11.4
    panEvaporation                (station, time) float64 162MB 0.1 0.1 ... 0.2
    evapotranspiration_shortCrop  (station, time) float64 162MB 0.19 ... 0.24
    ...                            ...
    frostCondition                (station, time) float64 162MB 0.0 0.0 ... 0.0
    heatCondition                 (station, time) float64 162MB 0.0 0.0 ... 15.0
    wind_3m_speed                 (station, time) float64 162MB 9.58 ... 4.34
    wind_3m_degN                  (station, time) float64 162MB 114.0 ... 146.0
    wind_10m_speed                (station, time) float64 162MB nan nan ... nan
    wind_10m_degN                 (station, time) float64 162MB nan nan ... nan
Attributes:
    time_zone:      UTC+08:00
    time_standard:  local_time
    comment:        All timestamps are local time in UTC+08:00 (Australia/Per...

In [16]:
ds.coords

Coordinates:
  * station  (station) <U22 17kB 'Floreat Park' 'Jerdacuttup' ... 'Gutha West'
  * time     (time) datetime64[ns] 842kB 2022-01-01 ... 2025-01-01T07:45:00
    lat      (station) float64 2kB -31.95 -33.77 -30.66 ... -31.74 -34.23 -29.06
    lon      (station) float64 2kB 115.8 120.5 116.0 122.7 ... 118.9 115.1 115.8
    code     (station) <U5 4kB 'FL' 'JP001' 'MZ' ... 'SB002' 'KR001' 'GU001'

## XR dataset to pd Dataframe

In [ ]:
Allanooka_ds= ds.sel(station= 'Allanooka')
df= Allanooka_ds.to_dataframe()
display(df)
df.to_csv("Allanooka.csv")


airTemperature  apparentAirTemperature  \
station      time                                                          
Floreat Park 2022-01-01 00:00:00            20.4                    18.3   
             2022-01-01 00:15:00            20.6                    18.8   
             2022-01-01 00:30:00            21.1                    19.1   
             2022-01-01 00:45:00            21.2                    19.1   
             2022-01-01 01:00:00            21.6                    19.6   
...                                          ...                     ...   
Gutha West   2025-01-01 06:45:00            39.7                    39.2   
             2025-01-01 07:00:00            39.5                    38.6   
             2025-01-01 07:15:00            40.5                    38.9   
             2025-01-01 07:30:00            40.4                    39.4   
             2025-01-01 07:45:00            39.4                    38.9   

                                  relativeHumidity  dewPoint  panEvaporation  \
station      time                                                              
Floreat Park 2022-01-01 00:00:00              48.9       9.3             0.1   
             2022-01-01 00:15:00              48.4       9.3             0.1   
             2022-01-01 00:30:00              47.3       9.4             0.2   
             2022-01-01 00:45:00              46.5       9.3             0.2   
             2022-01-01 01:00:00              45.8       9.4             0.2   
...                                            ...       ...             ...   
Gutha West   2025-01-01 06:45:00              18.9      11.7             0.3   
             2025-01-01 07:00:00              19.1      11.6             0.3   
             2025-01-01 07:15:00              18.1      11.6             0.4   
             2025-01-01 07:30:00              18.1      11.6             0.3   
             2025-01-01 07:45:00              18.9      11.4             0.2   

                                  evapotranspiration_shortCrop  \
station      time                                                
Floreat Park 2022-01-01 00:00:00                          0.19   
             2022-01-01 00:15:00                          0.19   
             2022-01-01 00:30:00                          0.21   
             2022-01-01 00:45:00                          0.22   
             2022-01-01 01:00:00                          0.23   
...                                                        ...   
Gutha West   2025-01-01 06:45:00                          0.33   
             2025-01-01 07:00:00                          0.36   
             2025-01-01 07:15:00                          0.43   
             2025-01-01 07:30:00                          0.34   
             2025-01-01 07:45:00                          0.24   

                                  evapotranspiration_tallCrop  \
station      time                                               
Floreat Park 2022-01-01 00:00:00                         0.26   
             2022-01-01 00:15:00                         0.25   
             2022-01-01 00:30:00                         0.28   
             2022-01-01 00:45:00                         0.31   
             2022-01-01 01:00:00                         0.31   
...                                                       ...   
Gutha West   2025-01-01 06:45:00                         0.43   
             2025-01-01 07:00:00                         0.50   
             2025-01-01 07:15:00                         0.63   
             2025-01-01 07:30:00                         0.49   
             2025-01-01 07:45:00                         0.32   

                                  richardsonUnits  solarExposure  rainfall  \
station      time                                                            
Floreat Park 2022-01-01 00:00:00            -0.25          496.7       0.0   
             2022-01-01 00:15:00            -0.25          549.2 

KeyboardInterrupt: 